# Employee Attrition Pipeline — Orchestration Workflow

**End-to-end DAG** that automates the full pipeline in a single run:

```
Task 1: Bronze Ingestion    (CSV → Delta)
    │
Task 2: dbt Transformations (Silver → Gold)
    │
    ├── Task 3a: Star Schema   (5 dims + 1 fact)
    └── Task 3b: ML Features   (gold_ml_features)
            │
        Task 4: ML Training     (3 models → MLflow)
            │
        Task 5: Batch Scoring   (predictions → Gold table)
```

### Usage
- **Databricks Community Edition:** Run this notebook manually — it calls each step sequentially.
- **Databricks Standard/Premium:** Import this logic into a **Databricks Workflow Job** with task dependencies for parallel execution.

### Notes
- Each task has validation checkpoints (row counts, schema checks)
- If a task fails, the pipeline stops with a clear error message
- Idempotent: safe to re-run (all tables use `overwrite` mode)

## Setup

In [ ]:
from datetime import datetime
import time

# Pipeline configuration
BRONZE_TABLE = "workspace.bronze.ibm_hr_employee_attrition"
SILVER_TABLE = "workspace.silver.silver_ibm_hr_cleaned"
GOLD_ML_FEATURES = "workspace.gold.gold_ml_features"
GOLD_ANALYTICS = "workspace.gold.gold_attrition_analytics"
GOLD_PREDICTIONS = "workspace.gold.gold_attrition_predictions"
GOLD_FACT = "workspace.gold.fact_employee_attrition"
EXPECTED_ROW_COUNT = 1470

pipeline_start = datetime.now()
task_results = {}

def log_task(task_name, status, details="", duration=0):
    """Log task result and store for final summary."""
    icon = "✅" if status == "PASS" else "❌"
    task_results[task_name] = {"status": status, "details": details, "duration": duration}
    print(f"{icon} [{status}] {task_name} ({duration:.1f}s) — {details}")

def validate_table(table_name, expected_min_rows=1):
    """Validate a Delta table exists and has rows. Returns row count."""
    count = spark.table(table_name).count()
    assert count >= expected_min_rows, f"{table_name}: expected >= {expected_min_rows} rows, got {count}"
    return count

print(f"Pipeline started: {pipeline_start.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'=' * 65}")

## Task 1 — Bronze Layer Ingestion
Load IBM HR CSV from DBFS into a Delta table.

In [ ]:
t_start = time.time()
task_name = "Task 1: Bronze Ingestion"
print(f"\n{'=' * 65}")
print(f"  {task_name}")
print(f"{'=' * 65}")

try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

    # Strategy: reuse existing Bronze table if it exists, otherwise load CSV
    table_exists = False
    try:
        existing_count = spark.table(BRONZE_TABLE).count()
        if existing_count >= EXPECTED_ROW_COUNT:
            table_exists = True
            print(f"  Bronze table already exists with {existing_count:,} rows — skipping CSV load")
    except Exception:
        pass

    if not table_exists:
        # Try multiple CSV source paths (Volumes first, then DBFS fallback)
        csv_paths = [
            "/Volumes/workspace/bronze/data/WA_Fn_UseC__HR_Employee_Attrition.csv",
            "/Volumes/workspace/default/data/WA_Fn_UseC__HR_Employee_Attrition.csv",
            "dbfs:/FileStore/WA_Fn_UseC__HR_Employee_Attrition.csv",
        ]

        df_raw = None
        for csv_path in csv_paths:
            try:
                df_raw = spark.read.format("csv") \
                    .option("header", "true") \
                    .option("inferSchema", "true") \
                    .load(csv_path)
                _ = df_raw.count()  # Force evaluation to check path works
                print(f"  CSV found at: {csv_path}")
                break
            except Exception:
                df_raw = None
                continue

        if df_raw is None:
            raise FileNotFoundError(
                "CSV not found. Upload it to a Unity Catalog Volume first:\n"
                "  1. In Databricks, go to Catalog > workspace > bronze\n"
                "  2. Create a Volume called 'data'\n"
                "  3. Upload WA_Fn_UseC__HR_Employee_Attrition.csv into it\n"
                "  Path should be: /Volumes/workspace/bronze/data/WA_Fn_UseC__HR_Employee_Attrition.csv"
            )

        raw_count = df_raw.count()
        raw_cols = len(df_raw.columns)
        print(f"  CSV loaded: {raw_count:,} rows x {raw_cols} columns")

        df_raw.write \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(BRONZE_TABLE)

    # Validate
    count = validate_table(BRONZE_TABLE, EXPECTED_ROW_COUNT)
    duration = time.time() - t_start
    log_task(task_name, "PASS", f"{count:,} rows", duration)

except Exception as e:
    duration = time.time() - t_start
    log_task(task_name, "FAIL", str(e), duration)
    raise


## Task 2 — Silver Layer Transformation
Clean, rename, encode columns via SQL (replicates dbt `silver_ibm_hr_cleaned`).

In [ ]:
t_start = time.time()
task_name = "Task 2: Silver Transformation"
print(f"\n{'=' * 65}")
print(f"  {task_name}")
print(f"{'=' * 65}")

try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
    
    silver_sql = f"""
    CREATE OR REPLACE TABLE {SILVER_TABLE} AS
    SELECT
        CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END AS attrition,
        Age AS age,
        CASE WHEN Gender = 'Male' THEN 1 ELSE 0 END AS gender,
        MaritalStatus AS marital_status,
        DistanceFromHome AS distance_from_home,
        Department AS department,
        JobRole AS job_role,
        JobLevel AS job_level,
        CASE WHEN OverTime = 'Yes' THEN 1 ELSE 0 END AS overtime,
        CASE 
            WHEN BusinessTravel = 'Non-Travel' THEN 0
            WHEN BusinessTravel = 'Travel_Rarely' THEN 1
            WHEN BusinessTravel = 'Travel_Frequently' THEN 2
        END AS business_travel_encoded,
        BusinessTravel AS business_travel,
        Education AS education,
        EducationField AS education_field,
        MonthlyIncome AS monthly_income,
        MonthlyRate AS monthly_rate,
        DailyRate AS daily_rate,
        HourlyRate AS hourly_rate,
        PercentSalaryHike AS percent_salary_hike,
        StockOptionLevel AS stock_option_level,
        TotalWorkingYears AS total_working_years,
        YearsAtCompany AS years_at_company,
        YearsInCurrentRole AS years_in_current_role,
        YearsSinceLastPromotion AS years_since_last_promotion,
        YearsWithCurrManager AS years_with_curr_manager,
        NumCompaniesWorked AS num_companies_worked,
        EnvironmentSatisfaction AS environment_satisfaction,
        JobSatisfaction AS job_satisfaction,
        RelationshipSatisfaction AS relationship_satisfaction,
        WorkLifeBalance AS work_life_balance,
        JobInvolvement AS job_involvement,
        PerformanceRating AS performance_rating,
        TrainingTimesLastYear AS training_times_last_year
    FROM {BRONZE_TABLE}
    """
    spark.sql(silver_sql)
    
    count = validate_table(SILVER_TABLE, EXPECTED_ROW_COUNT)
    cols = len(spark.table(SILVER_TABLE).columns)
    duration = time.time() - t_start
    log_task(task_name, "PASS", f"{count:,} rows, {cols} columns (dropped 4 constants + ID)", duration)

except Exception as e:
    duration = time.time() - t_start
    log_task(task_name, "FAIL", str(e), duration)
    raise

## Task 3 — Gold Layer: ML Features
Build the full ML feature table with engineered features, one-hot encoding, and aggregates.

In [ ]:
t_start = time.time()
task_name = "Task 3: Gold ML Features"
print(f"\n{'=' * 65}")
print(f"  {task_name}")
print(f"{'=' * 65}")

try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
    
    gold_ml_sql = f"""
    CREATE OR REPLACE TABLE {GOLD_ML_FEATURES} AS
    WITH silver AS (
        SELECT * FROM {SILVER_TABLE}
    ),
    engineered_features AS (
        SELECT
            *,
            monthly_income / NULLIF(total_working_years + 1, 0) AS income_per_year,
            monthly_income / NULLIF(age, 0) AS income_per_age,
            monthly_income / NULLIF(years_at_company + 1, 0) AS income_per_company_year,
            years_since_last_promotion / NULLIF(years_at_company + 1, 0) AS promo_stagnation,
            years_in_current_role / NULLIF(years_at_company + 1, 0) AS role_tenure_ratio,
            years_with_curr_manager / NULLIF(years_at_company + 1, 0) AS manager_stability,
            total_working_years / NULLIF(age, 0) AS career_ratio,
            years_at_company / NULLIF(total_working_years + 1, 0) AS company_loyalty_ratio,
            (environment_satisfaction + job_satisfaction + relationship_satisfaction) / 3.0 AS avg_satisfaction,
            (environment_satisfaction + job_satisfaction + relationship_satisfaction + work_life_balance) / 4.0 AS overall_wellbeing,
            CASE WHEN total_working_years <= 3 THEN 1 ELSE 0 END AS is_early_career,
            CASE WHEN years_at_company <= 2 THEN 1 ELSE 0 END AS is_new_hire,
            CASE WHEN years_since_last_promotion >= 5 THEN 1 ELSE 0 END AS long_time_since_promo,
            CASE WHEN age <= 30 THEN 1 ELSE 0 END AS is_young_employee,
            CASE WHEN job_level = 1 THEN 1 ELSE 0 END AS is_entry_level,
            CASE WHEN stock_option_level = 0 THEN 1 ELSE 0 END AS no_stock_options,
            CASE WHEN environment_satisfaction <= 2 THEN 1 ELSE 0 END AS low_env_satisfaction,
            CASE WHEN job_satisfaction <= 2 THEN 1 ELSE 0 END AS low_job_satisfaction,
            CASE WHEN work_life_balance <= 2 THEN 1 ELSE 0 END AS poor_work_life_balance,
            overtime * (CASE WHEN job_role = 'Sales Representative' THEN 1 ELSE 0 END) AS overtime_sales_rep,
            overtime * (CASE WHEN marital_status = 'Single' THEN 1 ELSE 0 END) AS overtime_single,
            (CASE WHEN job_level = 1 THEN 1 ELSE 0 END) * (CASE WHEN stock_option_level = 0 THEN 1 ELSE 0 END) AS entry_no_equity,
            distance_from_home * overtime AS distance_overtime_interaction
        FROM silver
    ),
    dept_aggregates AS (
        SELECT department,
            AVG(monthly_income) AS dept_avg_income,
            AVG(age) AS dept_avg_age,
            AVG(total_working_years) AS dept_avg_experience,
            COUNT(*) AS dept_headcount,
            AVG(overtime) AS dept_overtime_rate
        FROM silver GROUP BY department
    ),
    role_aggregates AS (
        SELECT job_role,
            AVG(monthly_income) AS role_avg_income,
            AVG(total_working_years) AS role_avg_experience,
            COUNT(*) AS role_headcount
        FROM silver GROUP BY job_role
    ),
    features_with_aggregates AS (
        SELECT ef.*,
            da.dept_avg_income, da.dept_avg_age, da.dept_avg_experience, da.dept_headcount, da.dept_overtime_rate,
            ra.role_avg_income, ra.role_avg_experience, ra.role_headcount,
            ef.monthly_income / NULLIF(da.dept_avg_income, 0) AS income_vs_dept_avg,
            ef.age / NULLIF(da.dept_avg_age, 0) AS age_vs_dept_avg
        FROM engineered_features ef
        LEFT JOIN dept_aggregates da ON ef.department = da.department
        LEFT JOIN role_aggregates ra ON ef.job_role = ra.job_role
    )
    SELECT
        attrition, age, distance_from_home, education, job_level,
        monthly_income, num_companies_worked, percent_salary_hike, stock_option_level,
        total_working_years, training_times_last_year, years_at_company,
        years_in_current_role, years_since_last_promotion, years_with_curr_manager,
        environment_satisfaction, job_satisfaction, job_involvement,
        performance_rating, relationship_satisfaction, work_life_balance,
        gender, overtime, business_travel_encoded,
        income_per_year, income_per_age, income_per_company_year,
        promo_stagnation, role_tenure_ratio, manager_stability,
        career_ratio, company_loyalty_ratio, avg_satisfaction, overall_wellbeing,
        is_early_career, is_new_hire, long_time_since_promo, is_young_employee,
        is_entry_level, no_stock_options, low_env_satisfaction, low_job_satisfaction,
        poor_work_life_balance, overtime_sales_rep, overtime_single,
        entry_no_equity, distance_overtime_interaction,
        dept_avg_income, dept_avg_age, dept_avg_experience, dept_headcount, dept_overtime_rate,
        role_avg_income, role_avg_experience, role_headcount,
        income_vs_dept_avg, age_vs_dept_avg,
        CASE WHEN department = 'Sales' THEN 1 ELSE 0 END AS dept_sales,
        CASE WHEN department = 'Research & Development' THEN 1 ELSE 0 END AS dept_research_dev,
        CASE WHEN department = 'Human Resources' THEN 1 ELSE 0 END AS dept_hr,
        CASE WHEN job_role = 'Sales Executive' THEN 1 ELSE 0 END AS role_sales_executive,
        CASE WHEN job_role = 'Research Scientist' THEN 1 ELSE 0 END AS role_research_scientist,
        CASE WHEN job_role = 'Laboratory Technician' THEN 1 ELSE 0 END AS role_lab_technician,
        CASE WHEN job_role = 'Manufacturing Director' THEN 1 ELSE 0 END AS role_manufacturing_director,
        CASE WHEN job_role = 'Healthcare Representative' THEN 1 ELSE 0 END AS role_healthcare_rep,
        CASE WHEN job_role = 'Manager' THEN 1 ELSE 0 END AS role_manager,
        CASE WHEN job_role = 'Sales Representative' THEN 1 ELSE 0 END AS role_sales_rep,
        CASE WHEN job_role = 'Research Director' THEN 1 ELSE 0 END AS role_research_director,
        CASE WHEN job_role = 'Human Resources' THEN 1 ELSE 0 END AS role_hr,
        CASE WHEN marital_status = 'Single' THEN 1 ELSE 0 END AS marital_single,
        CASE WHEN marital_status = 'Married' THEN 1 ELSE 0 END AS marital_married,
        CASE WHEN marital_status = 'Divorced' THEN 1 ELSE 0 END AS marital_divorced,
        CASE WHEN education_field = 'Life Sciences' THEN 1 ELSE 0 END AS edu_life_sciences,
        CASE WHEN education_field = 'Medical' THEN 1 ELSE 0 END AS edu_medical,
        CASE WHEN education_field = 'Marketing' THEN 1 ELSE 0 END AS edu_marketing,
        CASE WHEN education_field = 'Technical Degree' THEN 1 ELSE 0 END AS edu_technical,
        CASE WHEN education_field = 'Other' THEN 1 ELSE 0 END AS edu_other,
        CASE WHEN education_field = 'Human Resources' THEN 1 ELSE 0 END AS edu_hr
    FROM features_with_aggregates
    """
    spark.sql(gold_ml_sql)
    
    count = validate_table(GOLD_ML_FEATURES, EXPECTED_ROW_COUNT)
    cols = len(spark.table(GOLD_ML_FEATURES).columns)
    duration = time.time() - t_start
    log_task(task_name, "PASS", f"{count:,} rows, {cols} features", duration)

except Exception as e:
    duration = time.time() - t_start
    log_task(task_name, "FAIL", str(e), duration)
    raise

## Task 4 — Gold Layer: Star Schema for Power BI
Build 5 dimension tables + 1 fact table.

In [ ]:
t_start = time.time()
task_name = "Task 4: Gold Star Schema"
print(f"\n{'=' * 65}")
print(f"  {task_name}")
print(f"{'=' * 65}")

try:
    # Dimension: Department
    spark.sql(f"""
    CREATE OR REPLACE TABLE workspace.gold.dim_department AS
    WITH departments AS (SELECT DISTINCT department FROM {SILVER_TABLE})
    SELECT ROW_NUMBER() OVER (ORDER BY department) AS department_key,
           department AS department_name
    FROM departments
    """)
    print(f"  dim_department: {spark.table('workspace.gold.dim_department').count()} rows")
    
    # Dimension: Job Role
    spark.sql(f"""
    CREATE OR REPLACE TABLE workspace.gold.dim_job_role AS
    WITH job_roles AS (SELECT DISTINCT job_role, department FROM {SILVER_TABLE})
    SELECT ROW_NUMBER() OVER (ORDER BY department, job_role) AS job_role_key,
           job_role AS job_role_name,
           department AS department_name
    FROM job_roles
    """)
    print(f"  dim_job_role: {spark.table('workspace.gold.dim_job_role').count()} rows")
    
    # Dimension: Education Field
    spark.sql(f"""
    CREATE OR REPLACE TABLE workspace.gold.dim_education_field AS
    WITH edu_fields AS (SELECT DISTINCT education_field FROM {SILVER_TABLE})
    SELECT ROW_NUMBER() OVER (ORDER BY education_field) AS education_field_key,
           education_field AS education_field_name
    FROM edu_fields
    """)
    print(f"  dim_education_field: {spark.table('workspace.gold.dim_education_field').count()} rows")
    
    # Dimension: Marital Status
    spark.sql(f"""
    CREATE OR REPLACE TABLE workspace.gold.dim_marital_status AS
    WITH statuses AS (SELECT DISTINCT marital_status FROM {SILVER_TABLE})
    SELECT ROW_NUMBER() OVER (ORDER BY marital_status) AS marital_status_key,
           marital_status AS marital_status_name
    FROM statuses
    """)
    print(f"  dim_marital_status: {spark.table('workspace.gold.dim_marital_status').count()} rows")
    
    # Dimension: Business Travel
    spark.sql(f"""
    CREATE OR REPLACE TABLE workspace.gold.dim_business_travel AS
    WITH travel AS (SELECT DISTINCT business_travel FROM {SILVER_TABLE})
    SELECT ROW_NUMBER() OVER (ORDER BY business_travel) AS business_travel_key,
           business_travel AS business_travel_name,
           CASE
               WHEN business_travel = 'Non-Travel' THEN 0
               WHEN business_travel = 'Travel_Rarely' THEN 1
               WHEN business_travel = 'Travel_Frequently' THEN 2
           END AS travel_frequency_ordinal
    FROM travel
    """)
    print(f"  dim_business_travel: {spark.table('workspace.gold.dim_business_travel').count()} rows")
    
    # Fact Table
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD_FACT} AS
    WITH silver AS (SELECT * FROM {SILVER_TABLE}),
    dept AS (SELECT department_key, department_name FROM workspace.gold.dim_department),
    role AS (SELECT job_role_key, job_role_name, department_name FROM workspace.gold.dim_job_role),
    edu AS (SELECT education_field_key, education_field_name FROM workspace.gold.dim_education_field),
    marital AS (SELECT marital_status_key, marital_status_name FROM workspace.gold.dim_marital_status),
    travel AS (SELECT business_travel_key, business_travel_name FROM workspace.gold.dim_business_travel)
    SELECT
        ROW_NUMBER() OVER (ORDER BY s.department, s.job_role, s.age, s.monthly_income) AS employee_key,
        d.department_key, r.job_role_key, e.education_field_key, m.marital_status_key, t.business_travel_key,
        s.attrition,
        CASE WHEN s.attrition = 1 THEN 'Yes' ELSE 'No' END AS attrition_label,
        s.age,
        CASE
            WHEN s.age < 25 THEN 'Under 25'
            WHEN s.age BETWEEN 25 AND 34 THEN '25-34'
            WHEN s.age BETWEEN 35 AND 44 THEN '35-44'
            WHEN s.age BETWEEN 45 AND 54 THEN '45-54'
            ELSE '55+'
        END AS age_band,
        s.gender,
        CASE WHEN s.gender = 1 THEN 'Male' ELSE 'Female' END AS gender_label,
        s.distance_from_home, s.department, s.job_role, s.job_level, s.overtime,
        CASE WHEN s.overtime = 1 THEN 'Yes' ELSE 'No' END AS overtime_label,
        s.business_travel, s.business_travel_encoded,
        s.education,
        CASE
            WHEN s.education = 1 THEN 'Below College'
            WHEN s.education = 2 THEN 'College'
            WHEN s.education = 3 THEN 'Bachelor'
            WHEN s.education = 4 THEN 'Master'
            WHEN s.education = 5 THEN 'Doctor'
        END AS education_label,
        s.education_field,
        s.monthly_income,
        CASE
            WHEN s.monthly_income <= 2911 THEN 'Q1 (≤2,911)'
            WHEN s.monthly_income <= 4919 THEN 'Q2 (2,912-4,919)'
            WHEN s.monthly_income <= 8379 THEN 'Q3 (4,920-8,379)'
            ELSE 'Q4 (>8,379)'
        END AS income_quartile,
        s.monthly_rate, s.daily_rate, s.hourly_rate, s.percent_salary_hike, s.stock_option_level,
        s.total_working_years, s.years_at_company,
        CASE
            WHEN s.years_at_company <= 2 THEN '0-2 years'
            WHEN s.years_at_company <= 5 THEN '3-5 years'
            WHEN s.years_at_company <= 10 THEN '6-10 years'
            ELSE '10+ years'
        END AS tenure_band,
        s.years_in_current_role, s.years_since_last_promotion, s.years_with_curr_manager, s.num_companies_worked,
        s.environment_satisfaction, s.job_satisfaction, s.relationship_satisfaction,
        s.work_life_balance, s.job_involvement, s.performance_rating, s.training_times_last_year,
        ROUND(s.monthly_income / NULLIF(s.total_working_years, 0), 2) AS income_per_experience_year,
        ROUND(s.years_at_company * 1.0 / NULLIF(s.total_working_years, 0), 4) AS company_loyalty_ratio,
        s.years_at_company - s.years_in_current_role AS years_before_current_role
    FROM silver s
    LEFT JOIN dept d ON s.department = d.department_name
    LEFT JOIN role r ON s.job_role = r.job_role_name AND s.department = r.department_name
    LEFT JOIN edu e ON s.education_field = e.education_field_name
    LEFT JOIN marital m ON s.marital_status = m.marital_status_name
    LEFT JOIN travel t ON s.business_travel = t.business_travel_name
    """)
    
    count = validate_table(GOLD_FACT, EXPECTED_ROW_COUNT)
    duration = time.time() - t_start
    log_task(task_name, "PASS", f"5 dims + 1 fact ({count:,} rows)", duration)

except Exception as e:
    duration = time.time() - t_start
    log_task(task_name, "FAIL", str(e), duration)
    raise

## Task 5 — ML Model Training
Train 3 models, log to MLflow, select best by AUC-ROC.

In [ ]:
t_start = time.time()
task_name = "Task 5: ML Model Training"
print(f"\n{'=' * 65}")
print(f"  {task_name}")
print(f"{'=' * 65}")

try:
    import pandas as pd
    import mlflow
    import mlflow.sklearn
    import joblib, tempfile, os
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
    from mlflow.models.signature import infer_signature
    
    # CE workaround
    from mlflow.tracking._model_registry import utils as registry_utils
    registry_utils._get_registry_uri_from_spark_session = lambda: None
    mlflow.set_registry_uri("databricks-uc")
    
    EXPERIMENT_NAME = "/Users/mahmoudgribej7@gmail.com/employee-attrition-ibm-hr"
    mlflow.set_experiment(EXPERIMENT_NAME)
    mlflow.sklearn.autolog(disable=True)
    
    # Load data
    df = spark.table(GOLD_ML_FEATURES).toPandas()
    non_feature_cols = ["attrition"]
    string_cols = df.select_dtypes(include=["object"]).columns.tolist()
    feature_cols = [c for c in df.columns if c not in non_feature_cols + string_cols]
    X = df[feature_cols].fillna(0)
    y = df["attrition"]
    
    RANDOM_STATE = 42
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)
    
    print(f"  Data: {len(X_train)} train / {len(X_test)} test, {len(feature_cols)} features")
    
    # Define models
    models = {
        "Random Forest": {
            "model": RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_split=10,
                min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
            "use_scaled": False,
        },
        "Logistic Regression": {
            "model": LogisticRegression(C=0.1, penalty="l2", class_weight="balanced",
                max_iter=1000, random_state=RANDOM_STATE, solver="lbfgs"),
            "use_scaled": True,
        },
        "Gradient Boosted Trees": {
            "model": GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                min_samples_split=10, min_samples_leaf=5, subsample=0.8, random_state=RANDOM_STATE),
            "use_scaled": False,
        },
    }
    
    # Train & log each model
    results = {}
    for name, cfg in models.items():
        X_tr = X_train_scaled if cfg["use_scaled"] else X_train
        X_te = X_test_scaled if cfg["use_scaled"] else X_test
        
        cfg["model"].fit(X_tr, y_train)
        y_pred = cfg["model"].predict(X_te)
        y_prob = cfg["model"].predict_proba(X_te)[:, 1]
        
        metrics = {
            "auc_roc": roc_auc_score(y_test, y_prob),
            "f1": f1_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
        }
        results[name] = metrics
        
        with mlflow.start_run(run_name=name):
            mlflow.log_param("model_type", name)
            for k, v in metrics.items():
                mlflow.log_metric(k, v)
            mlflow.sklearn.log_model(cfg["model"], artifact_path="model")
            mlflow.log_text(",".join(feature_cols), "feature_columns.txt")
        
        print(f"  {name}: AUC={metrics['auc_roc']:.4f}, F1={metrics['f1']:.4f}, Recall={metrics['recall']:.4f}")
    
    # Find & log best model
    best_name = max(results, key=lambda k: results[k]["auc_roc"])
    best_model_obj = models[best_name]["model"]
    print(f"\n  Best model: {best_name} (AUC-ROC: {results[best_name]['auc_roc']:.4f})")
    
    with mlflow.start_run(run_name=f"BEST_{best_name}") as run:
        X_sample = X_test_scaled.head(5) if models[best_name]["use_scaled"] else X_test.head(5)
        signature = infer_signature(X_sample, best_model_obj.predict(X_sample))
        
        mlflow.log_param("model_type", best_name)
        for k, v in results[best_name].items():
            mlflow.log_metric(k, v)
        mlflow.sklearn.log_model(best_model_obj, artifact_path="model", signature=signature)
        mlflow.log_text(",".join(feature_cols), "feature_columns.txt")
        
        # Save scaler
        with tempfile.TemporaryDirectory() as tmpdir:
            scaler_path = os.path.join(tmpdir, "scaler.joblib")
            joblib.dump(scaler, scaler_path)
            mlflow.log_artifact(scaler_path, artifact_path="scaler")
        
        best_run_id = run.info.run_id
        print(f"  Best run ID: {best_run_id}")
    
    duration = time.time() - t_start
    log_task(task_name, "PASS", f"{best_name} (AUC {results[best_name]['auc_roc']:.4f})", duration)

except Exception as e:
    duration = time.time() - t_start
    log_task(task_name, "FAIL", str(e), duration)
    raise

## Task 6 — Batch Scoring
Score all employees, write risk predictions to Gold.

In [ ]:
t_start = time.time()
task_name = "Task 6: Batch Scoring"
print(f"\n{'=' * 65}")
print(f"  {task_name}")
print(f"{'=' * 65}")

try:
    import numpy as np
    import pandas as pd
    
    # Load best model from MLflow
    model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")
    scaler = joblib.load(mlflow.artifacts.download_artifacts(f"runs:/{best_run_id}/scaler/scaler.joblib"))
    
    with open(mlflow.artifacts.download_artifacts(f"runs:/{best_run_id}/feature_columns.txt"), 'r') as f:
        feature_cols = f.read().strip().split(',')
    
    # Score
    df = spark.table(GOLD_ML_FEATURES).toPandas()
    X = df[feature_cols].fillna(0)
    X_scaled = scaler.transform(X)
    
    probabilities = model.predict_proba(X_scaled)[:, 1]
    predictions = model.predict(X_scaled)
    print(f"  Scored {len(probabilities):,} employees")
    print(f"  Probability range: [{probabilities.min():.4f}, {probabilities.max():.4f}]")
    
    # Reconstruct labels
    df['department_name'] = np.where(
        df['dept_sales'] == 1, 'Sales',
        np.where(df['dept_research_dev'] == 1, 'Research & Development', 'Human Resources')
    )
    
    role_map = {
        'role_sales_executive': 'Sales Executive',
        'role_research_scientist': 'Research Scientist',
        'role_lab_technician': 'Laboratory Technician',
        'role_manufacturing_director': 'Manufacturing Director',
        'role_healthcare_rep': 'Healthcare Representative',
        'role_manager': 'Manager',
        'role_sales_rep': 'Sales Representative',
        'role_research_director': 'Research Director',
        'role_hr': 'Human Resources',
    }
    df['job_role_name'] = 'Unknown'
    for col, name in role_map.items():
        df.loc[df[col] == 1, 'job_role_name'] = name
    
    df['marital_status_name'] = np.where(
        df['marital_single'] == 1, 'Single',
        np.where(df['marital_married'] == 1, 'Married', 'Divorced')
    )
    df['overtime_label'] = df['overtime'].map({1: 'Yes', 0: 'No'})
    df['gender_label'] = df['gender'].map({1: 'Male', 0: 'Female'})
    
    # Attach predictions
    df['attrition_probability'] = np.round(probabilities, 4)
    df['predicted_attrition'] = predictions.astype(int)
    df['risk_tier'] = pd.cut(
        df['attrition_probability'],
        bins=[-0.01, 0.30, 0.60, 1.01],
        labels=['Low', 'Medium', 'High']
    ).astype(str)
    df['attrition_label'] = df['attrition'].map({1: 'Yes', 0: 'No'})
    
    # Build output
    output_df = df[[
        'department_name', 'job_role_name', 'age', 'gender_label',
        'marital_status_name', 'distance_from_home',
        'job_level', 'overtime_label', 'monthly_income',
        'years_at_company', 'total_working_years',
        'years_in_current_role', 'years_since_last_promotion',
        'environment_satisfaction', 'job_satisfaction',
        'work_life_balance', 'relationship_satisfaction',
        'attrition', 'attrition_label',
        'attrition_probability', 'predicted_attrition', 'risk_tier'
    ]].copy()
    
    output_df = output_df.sort_values(
        ['department_name', 'job_role_name', 'age', 'monthly_income']
    ).reset_index(drop=True)
    output_df.insert(0, 'employee_key', range(1, len(output_df) + 1))
    
    # Write to Gold
    output_spark_df = spark.createDataFrame(output_df)
    output_spark_df.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(GOLD_PREDICTIONS)
    
    count = validate_table(GOLD_PREDICTIONS, EXPECTED_ROW_COUNT)
    
    # Risk distribution summary
    risk_dist = output_df['risk_tier'].value_counts()
    print(f"  Risk distribution: High={risk_dist.get('High', 0)}, Medium={risk_dist.get('Medium', 0)}, Low={risk_dist.get('Low', 0)}")
    
    duration = time.time() - t_start
    log_task(task_name, "PASS", f"{count:,} predictions written", duration)

except Exception as e:
    duration = time.time() - t_start
    log_task(task_name, "FAIL", str(e), duration)
    raise

## Pipeline Summary

In [ ]:
pipeline_end = datetime.now()
total_duration = (pipeline_end - pipeline_start).total_seconds()

print(f"\n{'=' * 65}")
print(f"  PIPELINE EXECUTION SUMMARY")
print(f"{'=' * 65}")
print(f"  Started:  {pipeline_start.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Finished: {pipeline_end.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Duration: {total_duration:.1f}s ({total_duration/60:.1f} min)")
print(f"{'=' * 65}")

all_passed = True
for task, info in task_results.items():
    icon = "✅" if info['status'] == 'PASS' else '❌'
    print(f"  {icon} {task:35s}  {info['duration']:6.1f}s  {info['details']}")
    if info['status'] != 'PASS':
        all_passed = False

print(f"{'=' * 65}")
if all_passed:
    print(f"  ✅ ALL TASKS PASSED — Pipeline complete.")
    print(f"\n  Output tables ready for Power BI:")
    print(f"    • {GOLD_FACT}")
    print(f"    • workspace.gold.dim_department")
    print(f"    • workspace.gold.dim_job_role")
    print(f"    • workspace.gold.dim_education_field")
    print(f"    • workspace.gold.dim_marital_status")
    print(f"    • workspace.gold.dim_business_travel")
    print(f"    • {GOLD_PREDICTIONS}")
else:
    print(f"  ❌ PIPELINE FAILED — Check task errors above.")

print(f"{'=' * 65}")

---

## Databricks Workflow Job (Standard/Premium Tier)

If you have Databricks Standard or Premium, convert this to a **multi-task Workflow Job** for scheduled execution:

| Task | Notebook | Depends On |
|---|---|---|
| `bronze_ingestion` | `Bronze Layer Ingestion` | — |
| `silver_transform` | (this notebook, Task 2 cell) | `bronze_ingestion` |
| `gold_ml_features` | (this notebook, Task 3 cell) | `silver_transform` |
| `gold_star_schema` | (this notebook, Task 4 cell) | `silver_transform` |
| `ml_training` | `04_ml_model_training` | `gold_ml_features` |
| `batch_scoring` | `05_ml_batch_scoring` | `ml_training` |

This allows `gold_ml_features` and `gold_star_schema` to run **in parallel** after Silver completes.

### DAG Visualization

```
bronze_ingestion
       │
silver_transform
       │
   ┌───┴───┐
   │       │
gold_ml  gold_star
features  schema
   │
ml_training
   │
batch_scoring
```